### 2. Importações e Configurações Gerais

Nesta seção, importamos as bibliotecas e definimos as configurações para as APIs, modelos e arquivos.

In [ ]:
import pandas as pd
import json
import re
import os
import time
import sys
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from openai import OpenAI

# Importações corrigidas para a API do Google
from google import genai
from google.genai import types

# --- ARQUIVOS DE ENTRADA E SAÍDA ---
arquivo_excel = "microcontroladores-populares.xlsx"
arquivo_json_saida = "resultados_sem_rag.json"

# --- CONFIGURAÇÃO DA API GOOGLE GEMINI ---
# Substitua pela sua chave de API do Google Gemini
GEMINI_API_KEY = "SUA_API_AQUI"
client_gemini = genai.Client(api_key=GEMINI_API_KEY)
modelo_gemini = "gemini-2.5-flash"
# Modelo usado para extrair o valor das respostas (o mesmo em todas as categorias, com e sem RAG)
MODELO_EXTRACAO = "gemini-2.5-flash"

# --- TRATAMENTO DE ERROS DE API ---
# Chamadas que falham são repetidas com espera crescente. Se todas as tentativas falharem, a resposta é
# gravada com o prefixo ERRO_API: ela não é enviada para extração, é contada à parte na avaliação
# e é refeita automaticamente na próxima execução do script.
ERRO_API = "erro_api"
TENTATIVAS_API = 3
ESPERA_INICIAL_S = 5  # dobra a cada nova tentativa (5s, 10s, ...)

def eh_erro_api(texto):
    return str(texto).startswith(ERRO_API)

def chamar_com_retentativas(chamada, descricao):
    """Executa 'chamada' (função sem argumentos que retorna o texto da resposta ou levanta exceção).
    Retorna o texto, ou f"{ERRO_API}: <motivo>" se todas as tentativas falharem."""
    for tentativa in range(1, TENTATIVAS_API + 1):
        try:
            return chamada()
        except Exception as e:
            print(f"   [{descricao}] Falha na tentativa {tentativa}/{TENTATIVAS_API}: {e}")
            if tentativa < TENTATIVAS_API:
                time.sleep(ESPERA_INICIAL_S * 2 ** (tentativa - 1))
            else:
                return f"{ERRO_API}: {e}"

# --- CONFIGURAÇÃO DA API NVIDIA ---
# Substitua pela sua chave de API da NVIDIA
NVAPI_KEY = "SUA_API_AQUI"

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = NVAPI_KEY
)
# Nome do modelo NVIDIA a ser usado
modelo_nvidia = "openai/gpt-oss-20b"

# --- CONFIGURAÇÃO DA API CLOUDFLARE ---
# Substitua pelos seus dados da Cloudflare
CLOUDFLARE_API_BASE_URL = "https://api.cloudflare.com/client/v4/accounts/SEU_ACCOUNT_ID_AQUI/ai/run/"
headers_cloudflare = {"Authorization": "Bearer SUA_API_AQUI"}


# Lista de modelos Cloudflare a serem usados
modelos_cloudflare = [
    "@cf/meta/llama-4-scout-17b-16e-instruct",
    "@cf/mistralai/mistral-small-3.1-24b-instruct",
    "@cf/google/gemma-3-12b-it",
    "@cf/meta/llama-3.3-70b-instruct-fp8-fast"
]

# Configuração de retentativas para a API da Cloudflare
retry_strategy = Retry(
    total=3,  
    backoff_factor=1,  
    status_forcelist=[429, 500, 502, 503, 504],  
    allowed_methods=["POST"]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session_cloudflare = requests.Session()
session_cloudflare.mount("https://", adapter)

### 3. Funções de Interação com as APIs

In [ ]:
def generate_response_gemini(pergunta, usar_instrucao=True):
    """Gera uma resposta usando o modelo Google Gemini."""
    prompt_instrucao = "Responda a pergunta. Se não souber a resposta, retorne 'Não sei'.\n"
    prompt_completo = (prompt_instrucao + pergunta) if usar_instrucao else pergunta

    def chamada():
        response = client_gemini.models.generate_content(
            model=modelo_gemini,
            contents=prompt_completo,
            config=types.GenerateContentConfig(
                #temperature=0.2,
                thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
        )
        if not response.text:
            raise ValueError("a API não retornou texto")
        return response.text.strip()

    return chamar_com_retentativas(chamada, "Gemini (Geração)")

def extrair_valor_numerico_com_gemini(resposta, prompt_extracao):
    """Extrai valor numérico de uma resposta SEMPRE usando o Google Gemini."""
    # Respostas que falharam na geração não são enviadas para extração
    if eh_erro_api(resposta):
        return ERRO_API
    #if not resposta or resposta.lower().strip() in ["não sei", "nao sei"] or "erro na api" in resposta.lower():
    #    return "não sei"

    prompt = f"{prompt_extracao}\nExtraia o valor desta resposta: '{resposta}'"

    def chamada():
        response = client_gemini.models.generate_content(
            model=MODELO_EXTRACAO,
            contents=prompt,
            config=types.GenerateContentConfig(
                #temperature=0.0,
                thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
        )
        if not response.text:
            raise ValueError("a API não retornou texto")
        return response.text.strip().upper()

    valor_extraido = chamar_com_retentativas(chamada, "Gemini (Extração)")
    return ERRO_API if eh_erro_api(valor_extraido) else valor_extraido
def get_response_from_completion(completion):
    """Extrai a resposta do objeto de conclusão, verificando 'content' e 'reasoning_content'."""
    if not completion.choices:
        return None

    choice = completion.choices[0]
    message = choice.message

    # Tenta obter a resposta do campo 'content' primeiro
    content = message.content
    if content and content.strip():
        return content.strip()

    # Se 'content' estiver vazio, tenta obter do 'reasoning_content'
    reasoning_content = getattr(message, 'reasoning_content', None)
    if reasoning_content and reasoning_content.strip():
        print("[INFO] Resposta encontrada no campo 'reasoning_content'.")
        return reasoning_content.strip()

    # Se ambos estiverem vazios, retorna None
    return None

def generate_response_nvidia(pergunta, usar_instrucao=True):
    """Gera uma resposta usando o modelo da API da NVIDIA."""
    prompt_instrucao = "Responda a pergunta. Se não souber a resposta, retorne 'Não sei'. \nReasoning: None.\n"
    prompt_completo = (prompt_instrucao + pergunta) if usar_instrucao else pergunta

    def chamada():
        completion = client.chat.completions.create(
            model=modelo_nvidia,
            messages=[{"role": "user", "content": prompt_completo}],
            temperature=0.2,
            max_tokens=256,
            stream=False
        )
        resposta = get_response_from_completion(completion)
        if not resposta:
            raise ValueError("a API retornou uma resposta vazia em todos os campos")
        return resposta

    return chamar_com_retentativas(chamada, "NVIDIA")

def generate_response_cloudflare(model, pergunta, usar_instrucao=True):
    """Gera uma resposta usando um modelo da API da Cloudflare."""
    prompt_instrucao = "Responda a pergunta. Se não souber a resposta, retorne 'Não sei'.\n"
    prompt_completo = (prompt_instrucao + pergunta) if usar_instrucao else pergunta

    url = f"{CLOUDFLARE_API_BASE_URL}{model}"
    messages = [{"role": "user", "content": prompt_completo}]

    def chamada():
        response = session_cloudflare.post(url, headers=headers_cloudflare, json={"messages": messages}, timeout=30)
        response.raise_for_status()
        api_response = response.json()
        if 'result' in api_response and 'response' in api_response['result'] and str(api_response['result']['response']).strip():
            return api_response['result']['response'].strip()
        raise ValueError("resposta da Cloudflare vazia ou em formato inesperado")

    return chamar_com_retentativas(chamada, f"Cloudflare {model}")

### 4. Execução Principal e Coleta de Resultados

Este é o loop principal que lê o arquivo Excel, itera sobre as perguntas e modelos, coleta as respostas e salva os resultados em um arquivo JSON.

In [ ]:
def main():
    # Carrega o arquivo Excel
    try:
        df = pd.read_excel(arquivo_excel)
    except FileNotFoundError:
        print(f"ERRO: O arquivo '{arquivo_excel}' não foi encontrado. Verifique o nome e o local do arquivo.")
        sys.exit()

    # --- INÍCIO DA MODIFICAÇÃO: SELEÇÃO DE LINHAS E NOVOS PROMPTS ---
    # Defina a faixa de linhas que você deseja processar do arquivo Excel.
    # A contagem começa em 0. Ex: para processar da linha 1 a 5, use linha_inicial = 0 e linha_final = 5.
    linha_inicial = 0
    linha_final = len(df)   # Use len(df) para processar o arquivo inteiro

    df_selecionado = df.iloc[linha_inicial:linha_final]
    print(f"Processando {len(df_selecionado)} linhas (da linha {linha_inicial} à {linha_final-1}) do arquivo Excel.")

    # Defina as colunas de perguntas com os prompts de extração atualizados
    colunas_perguntas = [
        ("PERGUNTA01", "Flash Memory", "Extraia e retorne somente o valor numérico da resposta, com a sua unidade de medida em 'B', 'KB' ou 'MB'. "
                                     "Por exemplo, se a resposta for 'possui memória flash de 32kbytes', retorne '32KB' ou 'tem memória de 16 megabytes', retorne '16MB'. "
                                     "Caso não haja número ou a resposta for 'não sei', retorne 'não sei'"),
        ("PERGUNTA02", "Speed", "Extraia e retorne somente o valor numérico da resposta, com a sua unidade de medida em 'HZ', 'MHZ' ou 'GHZ'. "
                                "Por exemplo, se a resposta for 'possui clock de 4 megahertz', retorne '4MHZ' ou 'o clock do microcontrolador é de 16MHZ', retorne '16MHZ'. "
                                "Caso não haja número ou a resposta for 'não sei', retorne 'não sei'"),
        ("PERGUNTA03", "Number of I/O", "Extraia e retorne somente o número de entradas e saídas do microcontrolador, sem qualquer texto adicional, somente o número. "
                                        "Se a resposta se dividir em número de entradas e saídas, retorne o número total. "
                                        "Por exemplo, se a resposta for 'possui 32 entradas e 16 saídas', retorne '48' "
                                        "ou 'possui 16 entradas e saídas, sendo 8 delas configuráveis como analógicas', retorne '16'. "
                                        "Caso não haja número ou a resposta for 'não sei', retorne 'não sei'"),
        ("PERGUNTA04", "CommCAN", "Um LLM gerou uma resposta que diz se um microcontrolador possui ou não comunicação CANbus. "
                                  "Avalie a resposta e retorne 'sim' se o microcontrolador possui comunicação CANbus, "
                                  "ou 'não' se não possui. Se não souber, retorne 'não sei'"),
        ("PERGUNTA05", "CommI2C", "Um LLM gerou uma resposta que diz se um microcontrolador possui ou não comunicação I2C. "
                                  "Avalie a resposta e retorne 'sim' se o microcontrolador possui comunicação I2C, "
                                  "ou 'não' se não possui. Se não souber, retorne 'não sei'"),
        ("PERGUNTA06", "CommEthernet", "Um LLM gerou uma resposta que diz se um microcontrolador possui ou não comunicação Ethernet. "
                                        "Avalie a resposta e retorne 'sim' se o microcontrolador possui comunicação Ethernet, "
                                        "ou 'não' se não possui. Se não souber, retorne 'não sei'")
    ]
    # --- FIM DA MODIFICAÇÃO ---

    # Carrega resultados de execuções anteriores: respostas válidas são reaproveitadas e
    # respostas com erro de API são refeitas. Apague o arquivo de saída para começar do zero.
    todos_os_resultados = []
    if os.path.exists(arquivo_json_saida):
        with open(arquivo_json_saida, 'r', encoding='utf-8') as f:
            todos_os_resultados = json.load(f)
        print(f"Carregados {len(todos_os_resultados)} resultados de execuções anteriores.")
    existentes = {(r.get("linha_excel"), r.get("coluna_pergunta")): r for r in todos_os_resultados}

    modelos_a_testar = {
        'gemini': [modelo_gemini.replace('models/', '')],
        'nvidia': [modelo_nvidia],
        'cloudflare': modelos_cloudflare
    }
    nomes_esperados = [f"{nome_modelo}_sem_rag_{sufixo}"
                       for lista_de_modelos in modelos_a_testar.values() for nome_modelo in lista_de_modelos
                       for sufixo in ["com_instrucao", "sem_instrucao"]]

    print("\nIniciando o processo de perguntas e respostas...")

    for index, row in df_selecionado.iterrows():
        for pergunta_col, resposta_col, prompt_extracao in colunas_perguntas:
            pergunta = row.get(pergunta_col)
            resposta_verdadeira = row.get(resposta_col)

            if pd.isna(pergunta):
                continue
            
            resultado_pergunta = existentes.get((int(index), pergunta_col))
            if resultado_pergunta is None:
                resultado_pergunta = {
                    "linha_excel": int(index),
                    "pergunta": pergunta,
                    "coluna_pergunta": pergunta_col,
                    "resposta_verdadeira": str(resposta_verdadeira),
                    "coluna_resposta": resposta_col,
                    "respostas_modelos": []
                }
                todos_os_resultados.append(resultado_pergunta)

            respostas_validas = {r["modelo"]: r for r in resultado_pergunta["respostas_modelos"]
                                 if not eh_erro_api(r.get("resposta_completa")) and not eh_erro_api(r.get("valor_extraido"))}
            if all(nome in respostas_validas for nome in nomes_esperados):
                continue  # Pergunta já respondida por todos os modelos em execução anterior

            print("=" * 60)
            print(f"Linha {index} | Processando Pergunta: {pergunta}")
            novas_respostas = []

            for provider, lista_de_modelos in modelos_a_testar.items():
                for nome_modelo in lista_de_modelos:
                    # --- LOOP PARA COM/SEM INSTRUÇÃO ---
                    for com_instrucao in [True, False]:
                        sufixo = "com_instrucao" if com_instrucao else "sem_instrucao"
                        nome_resultado = f"{nome_modelo}_sem_rag_{sufixo}"
                        if nome_resultado in respostas_validas:
                            novas_respostas.append(respostas_validas[nome_resultado])
                            continue
                        print(f"\nTestando {nome_modelo} ({sufixo})...")

                        if provider == 'gemini':
                            resposta_completa = generate_response_gemini(pergunta, usar_instrucao=com_instrucao)
                        elif provider == 'nvidia':
                            resposta_completa = generate_response_nvidia(pergunta, usar_instrucao=com_instrucao)
                        elif provider == 'cloudflare':
                            resposta_completa = generate_response_cloudflare(nome_modelo, pergunta, usar_instrucao=com_instrucao)
                        
                        valor_extraido = extrair_valor_numerico_com_gemini(resposta_completa, prompt_extracao)
                        
                        novas_respostas.append({
                            "modelo": nome_resultado, # Nome do modelo com a categoria
                            "modelo_base": nome_modelo,
                            "rag": False,
                            "instrucao": com_instrucao,
                            "resposta_completa": resposta_completa,
                            "valor_extraido": valor_extraido,
                            "modelo_extracao": MODELO_EXTRACAO
                        })
                        print(f"  Resposta: {resposta_completa}")
                        print(f"  Valor Extraído (Gemini): {valor_extraido}")

            resultado_pergunta["respostas_modelos"] = novas_respostas
            
            with open(arquivo_json_saida, 'w', encoding='utf-8') as f:
                json.dump(todos_os_resultados, f, ensure_ascii=False, indent=4)

    erros = sum(eh_erro_api(r.get("resposta_completa")) or eh_erro_api(r.get("valor_extraido"))
                for item in todos_os_resultados for r in item["respostas_modelos"])
    if erros:
        print(f"\nATENÇÃO: {erros} respostas ficaram com erro de API. Rode esta célula novamente para refazê-las.")
    print("\nProcesso concluído! Resultados salvos em:", arquivo_json_saida)

main()

In [ ]:
import json
import plotly.graph_objects as go
import plotly.express as px

# Mesmo critério de acerto (Pint) usado no avalia_respostas_rag.py
from avalia_respostas_rag import classificar_resposta

# Nome do arquivo JSON
arquivo_json = "resultados_sem_rag.json"

# Carrega os dados do arquivo JSON
try:
    with open(arquivo_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print(f"Erro: O arquivo '{arquivo_json}' não foi encontrado.")
    exit()
except json.JSONDecodeError:
    print(f"Erro: Não foi possível decodificar o arquivo '{arquivo_json}'. Verifique se ele é um JSON válido.")
    exit()

# Inicializa dicionários para armazenar os dados de desempenho
model_performance = {}
erros_api = 0

# Itera sobre cada pergunta no arquivo
for question in data:
    resposta_verdadeira = question.get("resposta_verdadeira")
    model_responses = question["respostas_modelos"]

    for response in model_responses:
        model_name = response["modelo"]
        resultado = classificar_resposta(resposta_verdadeira, response.get("valor_extraido", ""), response.get("resposta_completa", ""))

        # Respostas com erro de API não são corretas nem erradas: ficam fora do gráfico
        if resultado == "erro_api":
            erros_api += 1
            continue

        if model_name not in model_performance:
            model_performance[model_name] = {"correct": 0, "incorrect": 0, "nao_sei": 0, "total": 0}

        if resultado == "nao_sei":
            model_performance[model_name]["nao_sei"] += 1
        elif resultado is True:
            model_performance[model_name]["correct"] += 1
        else:
            model_performance[model_name]["incorrect"] += 1

        model_performance[model_name]["total"] += 1

# Imprime o desempenho de cada modelo
print("---")
print("Performance dos Modelos:")
# ...existing code...
for model, stats in model_performance.items():
    correct = stats["correct"]
    incorrect = stats["incorrect"]
    nao_sei = stats["nao_sei"]
    total = stats["total"]
    correct_pct = (correct / total) * 100 if total > 0 else 0
    incorrect_pct = (incorrect / total) * 100 if total > 0 else 0
    nao_sei_pct = (nao_sei / total) * 100 if total > 0 else 0
    print(f"Modelo: {model}")
    print(f"  Certas: {correct} ({correct_pct:.2f}%) | Erradas: {incorrect} ({incorrect_pct:.2f}%) | Não sei: {nao_sei} ({nao_sei_pct:.2f}%)")
    print(f"  Total de respostas: {total}")
    print("-" * 30)
# ...existing code...

if erros_api:
    print(f"Respostas com erro de API (fora do gráfico): {erros_api}")

# Prepara os dados para o gráfico
models = list(model_performance.keys())

# Valores absolutos
corrects_abs = [model_performance[m]["correct"] for m in models]
incorrects_abs = [model_performance[m]["incorrect"] for m in models]
nao_seis_abs = [model_performance[m]["nao_sei"] for m in models]

# Valores percentuais
corrects_pct = [corrects_abs[i] / model_performance[m]["total"] * 100 if model_performance[m]["total"] > 0 else 0 for i, m in enumerate(models)]
incorrects_pct = [incorrects_abs[i] / model_performance[m]["total"] * 100 if model_performance[m]["total"] > 0 else 0 for i, m in enumerate(models)]
nao_seis_pct = [nao_seis_abs[i] / model_performance[m]["total"] * 100 if model_performance[m]["total"] > 0 else 0 for i, m in enumerate(models)]

# Cria o gráfico de barras empilhadas com Plotly
fig = go.Figure()

# Adiciona a barra de respostas corretas
fig.add_trace(go.Bar(
    x=models,
    y=corrects_pct,
    name='Corretas',
    marker_color='#5CB85C',
    text=[f'{corrects_abs[i]} ({p:.1f}%)' for i, p in enumerate(corrects_pct)],
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='white', size=11)
))

# Adiciona a barra de respostas erradas
fig.add_trace(go.Bar(
    x=models,
    y=incorrects_pct,
    name='Erradas',
    marker_color='#D9534F',
    text=[f'{incorrects_abs[i]} ({p:.1f}%)' for i, p in enumerate(incorrects_pct)],
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='white', size=11)
))

# Adiciona a barra de "não sei"
fig.add_trace(go.Bar(
    x=models,
    y=nao_seis_pct,
    name='Não sei',
    marker_color='#A9A9A9',
    text=[f'{nao_seis_abs[i]} ({p:.1f}%)' for i, p in enumerate(nao_seis_pct)],
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='black', size=11)
))

# Configura o layout do gráfico
fig.update_layout(
    barmode='stack',
    title={
        'text': 'Desempenho dos Modelos (Corretas, Erradas, Não sei)',
        'y':0.9,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title='Modelo',
    yaxis_title='Percentual de Respostas (%)',
    legend_title='Legenda',
    font=dict(
        family="Arial, sans-serif",
        size=12,
        color="#7f7f7f"
    ),
    hovermode="x unified",
    template='plotly_white'
)

# Exibe e salva o gráfico interativo
fig.write_html("performance_sem_rag.html")
print("---")
print("Gráfico de desempenho empilhado gerado e salvo como 'performance_sem_rag.html'!")
print("---")

### 5. Análise e Visualização dos Resultados

Após a coleta, este código carrega o arquivo JSON e gera um gráfico comparativo do desempenho de cada modelo.